# Pancreatic Cancer Risk Prediction Model

This project implements a machine learning approach to predict the risk of pancreatic cancer. Early detection is a critical medical challenge, and our objective is to build a robust diagnostic support tool that prioritizes patient safety above all else.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn.utils.parallel")

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

## 1. Data Preparation and Cleaning
In this phase, we prepare the dataset for analysis. We remove irrelevant identifiers (Patient_ID) and leakage-prone features (e.g., survival time, which would not be available at the time of diagnosis). We also map categorical variables like cancer stages and tumor grades to numerical values, ensuring the data is ready for the preprocessing pipeline.

In [2]:
# 1. LOAD AND CLEAN DATA
df = pd.read_csv("pancreatic_cancer_dataset_cleaned.csv")
leakage_and_id_cols = ['Patient_ID', 'Diagnosis_Year', 'Diagnosis_Date', 'Survival_Months', 'Five_Year_Status']
df = df.drop(columns=[c for c in leakage_and_id_cols if c in df.columns])

df['Cancer_Stage'] = df['Cancer_Stage'].map({'Stage I': 1, 'Stage II': 2, 'Stage III': 3, 'Stage IV': 4})
df['Tumor_Grade'] = df['Tumor_Grade'].map({'Grade 1 (Well Differentiated)': 1, 'Grade 2 (Moderately Differentiated)': 2,
                                           'Grade 3 (Poorly Differentiated)': 3})

df = df.rename(columns={'Survived': 'Target'}).dropna(subset=['Target'])
X = df.drop(columns=['Target'])
y = df['Target']

In [3]:
# 2. PREPROCESSING
num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()

preprocessor = ColumnTransformer([
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), num_cols),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
                      ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), cat_cols)
])

## 2. Methodology and Model Architecture
We employ a `RandomForestClassifier` within an automated `Pipeline`. To mitigate the impact of class imbalance (as high-risk cases are typically less frequent than low-risk ones), we utilize `class_weight='balanced'`. This ensures that the model assigns higher importance to the minority class, effectively improving the detection rate of high-risk patients.

In [ ]:
# 3. MODEL PIPELINE
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=29, n_jobs=-1, class_weight="balanced"))
])

param_dist = {
    'classifier__n_estimators': [100, 200, 500],
    'classifier__max_depth': [None, 10, 20, 30],
    'classifier__min_samples_split': [2, 5, 20],
    'classifier__min_samples_leaf': [1, 2, 4, 10],
    'classifier__max_features': ['sqrt', 'log2', 0.2, 0.5],
    'classifier__criterion': ['gini', 'entropy', 'log_loss']
}

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=29)

random_search = RandomizedSearchCV(
    pipeline,
    param_dist,
    n_iter=20,
    scoring='recall',
    cv=5,
    n_jobs=-1,
    random_state=29,
    verbose=0
)
random_search.fit(X_train, y_train)

### 3. Cost-Sensitive Optimization Strategy
In medical diagnostics, the value of a human life is absolute and cannot be quantified. However, to translate clinical priorities into an algorithmic framework, we must define a clear cost-benefit structure. We have implemented a cost-sensitive learning approach where a **False Negative (FN)**—the failure to detect a high-risk patient—is assigned a hypothetical, critical penalty of **100,000,000 units**. This reflects our fundamental priority: to ensure that no patient at risk is missed, regardless of the consequences. 

In contrast, we assign a penalty of **10,000 units** to a **False Positive (FP)**. While false alarms carry a burden of unnecessary follow-up procedures and emotional stress, they are clinically manageable compared to the irreversible outcome of a missed diagnosis. By optimizing the decision threshold against these specific weights, we force the model to prioritize sensitivity (Recall) above all else.

In [ ]:
# 4. COST-OPTIMIZED THRESHOLD SEARCH
COST_FN = 100_000_000  # Cost for False Negative
COST_FP = 10_000       # Cost for False Positive

y_probs = random_search.best_estimator_.predict_proba(X_test)[:, 1]
thresholds = np.linspace(0.01, 0.99, 100)

best_threshold = 0.5
min_total_cost = float('inf')

for t in thresholds:
    y_pred = (y_probs >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    total_cost = (fn * COST_FN) + (fp * COST_FP)

    if total_cost < min_total_cost:
        min_total_cost = total_cost
        best_threshold = t

In [ ]:
# 5. FINAL RESULTS WITH OPTIMAL THRESHOLD
y_pred_final = (y_probs >= best_threshold).astype(int)
cm = confusion_matrix(y_test, y_pred_final)
tn, fp, fn, tp = cm.ravel()

print(f"\n--- OPTIMIZED RESULTS ---")
print(f"Best Threshold: {best_threshold:.2f}")
print(f"Minimum Total Cost: {min_total_cost:_}")
print(f"TN: {tn}, FP: {fp}, FN: {fn}, TP: {tp}")

In [ ]:
# Confusion Matrix Plot
fig, ax = plt.subplots(figsize=(6, 6))
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Survived', 'High Risk']).plot(ax=ax, cmap='Blues')
plt.title(f'Cost-Optimized Confusion Matrix (Threshold {best_threshold:.2f})')
plt.show()

## 4. Results and Performance Analysis
The model's performance shows that our cost-optimization strategy successfully prioritized sensitivity. However, we observe **41 False Positives (FP)** in the test set. While this maintains a high safety margin (ensuring the lowest possible number of missed high-risk cases), these false alarms represent an area where the model’s precision can be further refined.

In [ ]:
# 6. RISK ANALYSIS FOR 10 SAMPLES
print("\n--- RISK PREDICTION FOR 10 SAMPLES ---")
samples = X_test.iloc[:10]
sample_probs = random_search.best_estimator_.predict_proba(samples)[:, 1]

for i, prob in enumerate(sample_probs, start=1):
    risk_percentage = prob * 100
    status = "HIGH RISK" if prob >= best_threshold else "LOW RISK"
    alert = " !!! WARNING: CRITICAL RISK !!!" if risk_percentage >= 80 else ""

    print(f"Patient {i}: Risk {risk_percentage:.2f}% | Status: {status}{alert}")

## 5. Future Work and Optimization Roadmap
To reduce False Positives to zero without compromising safety, our future roadmap includes:
- **Advanced Feature Engineering:** Creating domain-specific features that capture deeper physiological correlations.
- **Enhanced Feature Selection:** Rigorous analysis of feature importance to remove "noise" that leads to misclassifications.
- **Model Evolution:** Beyond the current Random Forest implementation, we plan to experiment with Gradient Boosting architectures (such as XGBoost or LightGBM) to refine decision boundaries and improve overall accuracy.

## 6. Conclusion
This project demonstrates that automated medical diagnostic tools can be highly effective when grounded in a sound cost-sensitive framework. By aligning our model's performance with clinical priorities, we have created a robust foundation for supporting early pancreatic cancer risk assessment, with a clear path forward for continuous iteration and improvement.